In [13]:
!pip install -q -U langchain langchain-groq gspread pydantic

In [14]:
import os
from google.colab import userdata

# Get Groq API key from Colab Secrets
api_key = userdata.get("GROQ_API_KEY")

if not api_key:
    raise ValueError("GROQ_API_KEY not found in Colab Secrets!")

os.environ["GROQ_API_KEY"] = api_key

from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.2,
    max_retries=3
)

print("TravelEase AI is ready!")

TravelEase AI is ready!


In [15]:
SYSTEM = """
ROLE:
You are TravelEase's friendly AI travel booking assistant.

BUSINESS:
TravelEase is a travel agency that helps customers explore
holiday packages and make booking enquiries.

AVAILABLE PACKAGES:
1. Goa - 3 Days - Rs 5000 per person
2. Kerala - 4 Days - Rs 8000 per person
3. Manali - 5 Days - Rs 12000 per person
4. Jaipur - 3 Days - Rs 6000 per person

BOOKING RULES:
1. Package prices are per person.
2. Customers must book at least 48 hours before travel.
3. Collect customer name, destination, travel date,
   and number of travellers.
4. Never confirm a booking without actual verification.
5. Never invent hotel, flight or transport availability.
6. If availability information is unavailable, say:
   "I'll check with our travel team."
7. Save a booking enquiry only when all required details
   are available and the booking notice rule is satisfied.
8. Saving an enquiry does NOT mean the booking is confirmed.

TOOLS:
- get_package_price: Check the actual package price.
- check_booking_notice: Check the 48-hour booking rule.
- log_booking_enquiry: Save a complete enquiry in Google Sheets.

PRICING:
Total price = Price per person x Number of travellers.

RESPONSE STYLE:
- Be friendly and helpful.
- Keep answers simple and clear.
- Reply in English or Telugu according to the customer.
- Ask for missing details instead of guessing.
"""
print(SYSTEM)


ROLE:
You are TravelEase's friendly AI travel booking assistant.

BUSINESS:
TravelEase is a travel agency that helps customers explore
holiday packages and make booking enquiries.

AVAILABLE PACKAGES:
1. Goa - 3 Days - Rs 5000 per person
2. Kerala - 4 Days - Rs 8000 per person
3. Manali - 5 Days - Rs 12000 per person
4. Jaipur - 3 Days - Rs 6000 per person

BOOKING RULES:
1. Package prices are per person.
2. Customers must book at least 48 hours before travel.
3. Collect customer name, destination, travel date,
   and number of travellers.
4. Never confirm a booking without actual verification.
5. Never invent hotel, flight or transport availability.
6. If availability information is unavailable, say:
   "I'll check with our travel team."
7. Save a booking enquiry only when all required details
   are available and the booking notice rule is satisfied.
8. Saving an enquiry does NOT mean the booking is confirmed.

TOOLS:
- get_package_price: Check the actual package price.
- check_book

In [16]:
from pydantic import BaseModel, Field
from typing import Optional, Literal

class BookingEnquiry(BaseModel):

    customer_name: Optional[str] = Field(
        default=None,
        description="Full name of the customer"
    )

    destination: Optional[
        Literal["Goa", "Kerala", "Manali", "Jaipur"]
    ] = Field(
        default=None,
        description="Selected holiday destination"
    )

    travel_date: Optional[str] = Field(
        default=None,
        description="Travel date in YYYY-MM-DD format"
    )

    travellers: Optional[int] = Field(
        default=None,
        ge=1,
        description="Number of people travelling; must be at least 1"
    )

print("BookingEnquiry model created!")

BookingEnquiry model created!


In [18]:
extractor = llm.with_structured_output(BookingEnquiry)

customer_messages = [
    "Hi, I'm Samhitha. I want to visit Goa on 2026-12-20 with 2 people.",

    "Book Kerala for me and my family. We are 4 members. My name is Ravi. Date is 2026-12-25.",

    "Hello, this is Priya. Planning a Manali trip on 2026-12-30.",

    "We are three friends. Want Jaipur trip on 2026-12-28. I'm Arjun.",

    "Hey, I want to travel to Goa. My name is Sneha."
]

extracted_records = []

for i, message in enumerate(customer_messages, start=1):

    print(f"\nCUSTOMER MESSAGE {i}:")
    print(message)

    result = extractor.invoke(message)

    extracted_records.append(result)

    print("\nSTRUCTURED OUTPUT:")
    print(result.model_dump())


CUSTOMER MESSAGE 1:
Hi, I'm Samhitha. I want to visit Goa on 2026-12-20 with 2 people.

STRUCTURED OUTPUT:
{'customer_name': 'Samhitha', 'destination': 'Goa', 'travel_date': '2026-12-20', 'travellers': 2}

CUSTOMER MESSAGE 2:
Book Kerala for me and my family. We are 4 members. My name is Ravi. Date is 2026-12-25.

STRUCTURED OUTPUT:
{'customer_name': 'Ravi', 'destination': 'Kerala', 'travel_date': '2026-12-25', 'travellers': 4}

CUSTOMER MESSAGE 3:
Hello, this is Priya. Planning a Manali trip on 2026-12-30.

STRUCTURED OUTPUT:
{'customer_name': 'Priya', 'destination': 'Manali', 'travel_date': '2026-12-30', 'travellers': None}

CUSTOMER MESSAGE 4:
We are three friends. Want Jaipur trip on 2026-12-28. I'm Arjun.

STRUCTURED OUTPUT:
{'customer_name': 'Arjun', 'destination': 'Jaipur', 'travel_date': '2026-12-28', 'travellers': 3}

CUSTOMER MESSAGE 5:
Hey, I want to travel to Goa. My name is Sneha.

STRUCTURED OUTPUT:
{'customer_name': 'Sneha', 'destination': 'Goa', 'travel_date': None, 't

In [19]:
from langchain.tools import tool

PACKAGES = {
    "goa": {
        "days": 3,
        "price": 5000
    },
    "kerala": {
        "days": 4,
        "price": 8000
    },
    "manali": {
        "days": 5,
        "price": 12000
    },
    "jaipur": {
        "days": 3,
        "price": 6000
    }
}


@tool
def get_package_price(destination: str) -> str:
    """
    Get the actual per-person holiday package price and duration
    for Goa, Kerala, Manali or Jaipur.
    Use this tool whenever a customer asks about package pricing.
    """

    package = PACKAGES.get(destination.lower().strip())

    if not package:
        return "Destination not available. Please choose Goa, Kerala, Manali or Jaipur."

    return (
        f"{destination.title()} package: "
        f"{package['days']} days, "
        f"Rs {package['price']} per person."
    )


from datetime import datetime
from zoneinfo import ZoneInfo


def check_notice(date_text: str) -> str:

    try:
        travel_date = datetime.strptime(
            date_text, "%Y-%m-%d"
        ).date()

        today = datetime.now(
            ZoneInfo("Asia/Kolkata")
        ).date()

        days_left = (travel_date - today).days

        if days_left < 2:
            return (
                "Booking notice is less than 48 hours "
                "or the travel date is not in the future. "
                "Booking cannot proceed under the stated policy."
            )

        return (
            f"Booking notice requirement is satisfied. "
            f"There are {days_left} calendar days until travel. "
            "Final availability still needs travel-team verification."
        )

    except ValueError:
        return "Invalid date. Please use YYYY-MM-DD format."


@tool
def check_booking_notice(travel_date: str) -> str:
    """
    Check whether a customer's travel date satisfies
    TravelEase's minimum 48-hour advance booking policy.
    The date must be in YYYY-MM-DD format.
    """

    return check_notice(travel_date)


print(get_package_price.invoke({"destination": "Goa"}))
print(check_booking_notice.invoke({"travel_date": "2026-12-20"}))

Goa package: 3 days, Rs 5000 per person.
Booking notice requirement is satisfied. There are 24 calendar days until travel. Final availability still needs travel-team verification.


In [20]:
import gspread
from google.colab import auth
import google.auth

auth.authenticate_user()

credentials, project = google.auth.default()

gc = gspread.authorize(credentials)

SHEET_NAME = "TravelEase Booking Enquiries"

# Open existing sheet or create a new one
try:
    spreadsheet = gc.open(SHEET_NAME)
    print("Existing Google Sheet opened!")

except gspread.SpreadsheetNotFound:
    spreadsheet = gc.create(SHEET_NAME)
    print("New Google Sheet created!")

worksheet = spreadsheet.sheet1

headers = [
    "Customer Name",
    "Destination",
    "Travel Date",
    "Travellers",
    "Price Per Person",
    "Total Price",
    "Enquiry Status"
]

# Add headers only if the sheet is empty
if not worksheet.get_all_values():
    worksheet.append_row(headers)

print("Google Sheets connected successfully!")
print("Sheet URL:", spreadsheet.url)

New Google Sheet created!
Google Sheets connected successfully!
Sheet URL: https://docs.google.com/spreadsheets/d/1eBXaJjF0siiC7P-LaoHBSBddrvxYHEP8dCpM1FBcXt0


In [21]:
@tool
def log_booking_enquiry(
    customer_name: str,
    destination: str,
    travel_date: str,
    travellers: int
) -> str:
    """
    Validate a complete TravelEase booking enquiry, calculate
    the total package price and save it to Google Sheets.
    Use only when customer name, destination, travel date
    and number of travellers are available.
    This records an enquiry, not a confirmed booking.
    """

    try:
        # Validate all customer details
        record = BookingEnquiry(
            customer_name=customer_name,
            destination=destination.title(),
            travel_date=travel_date,
            travellers=travellers
        )

        if not all([
            record.customer_name,
            record.destination,
            record.travel_date,
            record.travellers
        ]):
            return "Missing booking details. Please collect all required information."

        # Validate booking notice
        notice_result = check_notice(record.travel_date)

        if "requirement is satisfied" not in notice_result:
            return notice_result

        # Get actual package price
        package = PACKAGES.get(record.destination.lower())

        if not package:
            return "Destination is not available."

        price = package["price"]
        total = price * record.travellers

        # Save enquiry in Google Sheets
        worksheet.append_row([
            record.customer_name,
            record.destination,
            record.travel_date,
            record.travellers,
            price,
            total,
            "Pending travel-team verification"
        ])

        return (
            f"Booking enquiry saved successfully!\n"
            f"Customer: {record.customer_name}\n"
            f"Destination: {record.destination}\n"
            f"Travellers: {record.travellers}\n"
            f"Total package price: Rs {total}\n"
            f"Status: Pending travel-team verification."
        )

    except Exception as e:
        return f"Unable to save enquiry: {str(e)}"


print("Booking enquiry tool created!")

Booking enquiry tool created!


In [22]:
from langchain.agents import create_agent

tools = [
    get_package_price,
    check_booking_notice,
    log_booking_enquiry
]

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=SYSTEM
)

print("TravelEase Agent created successfully!")

TravelEase Agent created successfully!


In [23]:
def ask_travelease(question):

    result = agent.invoke({
        "messages": [
            {
                "role": "user",
                "content": question
            }
        ]
    })

    return result


response = ask_travelease(
    "Hi, my name is Samhitha. "
    "I want to visit Goa on 2026-12-20 with 2 travellers. "
    "Please check the package price and save my booking enquiry."
)

print(response["messages"][-1].content)

Hi Samhitha! 🎉

Your booking enquiry for Goa on 2026‑12‑20 with 2 travellers has been saved.  
- **Package:** Goa – 3 days  
- **Price per person:** ₹5,000  
- **Total:** ₹10,000  

We’re now waiting for the travel team to confirm availability. I’ll keep you posted as soon as we have an update. If you have any other questions or need help with anything else, just let me know!


In [24]:
questions = [
    "What holiday packages do you offer?",
    "How much does a Kerala package cost?",
    "Can I book Manali with 3 people?",
    "What details do you need to save my booking enquiry?"
]

for question in questions:

    print("\nCustomer:", question)

    result = ask_travelease(question)

    print("TravelEase:", result["messages"][-1].content)


Customer: What holiday packages do you offer?
TravelEase: Sure! Here are the holiday packages we offer:

| Destination | Duration | Price (per person) |
|-------------|----------|--------------------|
| Goa         | 3 days   | ₹5,000 |
| Kerala      | 4 days   | ₹8,000 |
| Manali      | 5 days   | ₹12,000 |
| Jaipur      | 3 days   | ₹6,000 |

Let me know which one interests you, and we can dive into dates, traveller details, and any special requests!

Customer: How much does a Kerala package cost?
TravelEase: Kerala package costs Rs 8,000 per person for a 4‑day stay. Let me know if you’d like to book or need more details!

Customer: Can I book Manali with 3 people?
TravelEase: Sure thing! To help you with the Manali booking for 3 people, could you please let me know:

1. Your full name (for the booking record).  
2. The date you’d like to travel (in YYYY‑MM‑DD format).

Once I have those details, I’ll check the 48‑hour rule and calculate the total price for you.

Customer: What deta

In [25]:
@tool("tool1")
def tool1(destination: str) -> str:
    """does stuff"""
    return get_package_price.invoke({
        "destination": destination
    })


# Create an experimental agent with the poorly named tool
bad_agent = create_agent(
    model=llm,
    tools=[
        tool1,
        check_booking_notice,
        log_booking_enquiry
    ],
    system_prompt=SYSTEM
)

experiment = bad_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What is the price of the Goa package?"
        }
    ]
})

# Check which tool the model selected
for message in experiment["messages"]:

    if getattr(message, "tool_calls", None):

        for call in message.tool_calls:
            print("Selected tool:", call["name"])

print("\nFinal response:")
print(experiment["messages"][-1].content)


Final response:
The Goa package costs **₹5,000 per person** (for a 3‑day stay).


In [26]:
agent = create_agent(
    model=llm,
    tools=[
        get_package_price,
        check_booking_notice,
        log_booking_enquiry
    ],
    system_prompt=SYSTEM
)

print("Original TravelEase tools restored!")

Original TravelEase tools restored!


In [27]:
trace_result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": (
                "Are you open for bookings under your 48-hour policy? "
                "What is the price of 2 Goa packages? "
                "My name is Ravi and my travel date is 2026-12-20."
            )
        }
    ]
})

# Print complete trace
for message in trace_result["messages"]:
    message.pretty_print()


# Count tool calls
tool_call_count = 0

for message in trace_result["messages"]:

    calls = getattr(message, "tool_calls", [])

    tool_call_count += len(calls)


# Count model calls
from langchain_core.messages import AIMessage

model_call_count = sum(
    isinstance(message, AIMessage)
    for message in trace_result["messages"]
)

print("\n========== TRACE SUMMARY ==========")

print("Total tool calls:", tool_call_count)
print("Total model calls:", model_call_count)

print("\nFinal Answer:")
print(trace_result["messages"][-1].content)

================================ Human Message =================================

Are you open for bookings under your 48-hour policy? What is the price of 2 Goa packages? My name is Ravi and my travel date is 2026-12-20.
================================== Ai Message ==================================
Tool Calls:
  check_booking_notice (fc_64baeff9-1745-4189-a2e2-153aa646fc2a)
 Call ID: fc_64baeff9-1745-4189-a2e2-153aa646fc2a
  Args:
    travel_date: 2026-12-20
================================= Tool Message =================================
Name: check_booking_notice

Booking notice requirement is satisfied. There are 24 calendar days until travel. Final availability still needs travel-team verification.
================================== Ai Message ==================================
Tool Calls:
  get_package_price (fc_faf53737-3025-4ace-ab3e-c6bb8afffa5e)
 Call ID: fc_faf53737-3025-4ace-ab3e-c6bb8afffa5e
  Args:
    destination: Goa
================================= Tool Message =====

In [28]:
records = worksheet.get_all_records()

for record in records:
    print(record)